# Tests de integración de `infer_trips_from_traces()`

## Sección 0. Preparación

### 0.1 Imports generales

In [1]:
import json
from copy import deepcopy

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal

### 0.2 Imports del módulo

In [2]:
from pylondrina.datasets import TraceDataset, TripDataset
from pylondrina.errors import InferenceError
from pylondrina.importing_traces import ImportTraceOptions, import_traces_from_dataframe
from pylondrina.reports import InferenceReport
from pylondrina.schema import DomainSpec, FieldSpec, TraceSchema, TripSchema
from pylondrina.transforms.inference import InferTripsOptions, infer_trips_from_traces
from pylondrina.validation_traces import TraceValidationOptions, validate_traces

### 0.3 Helpers de testing reutilizables

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")


def issue_codes(report_or_issues):
    issues = report_or_issues.issues if hasattr(report_or_issues, "issues") else report_or_issues
    return [issue.code for issue in issues]


def assert_has_code(report_or_issues, code: str):
    codes = issue_codes(report_or_issues)
    assert code in codes, f"No se encontró {code}. Codes emitidos: {codes}"


def clone_tracedataset(traces: TraceDataset) -> TraceDataset:
    return TraceDataset(
        data=traces.data.copy(deep=True),
        schema=traces.schema,
        provenance=deepcopy(traces.provenance),
        metadata=deepcopy(traces.metadata),
    )

### 0.4 Schemas reutilizables

In [4]:
TRACE_MIN_FIELDS = ["point_id", "user_id", "time_utc", "latitude", "longitude"]
TRIP_MIN_FIELDS = [
    "movement_id",
    "user_id",
    "origin_longitude",
    "origin_latitude",
    "destination_longitude",
    "destination_latitude",
    "origin_time_utc",
    "destination_time_utc",
    "origin_h3_index",
    "destination_h3_index",
    "trip_id",
    "movement_seq",
]


def make_trace_schema_rich() -> TraceSchema:
    fields = {
        "point_id": FieldSpec(name="point_id", dtype="string", required=True, constraints={"unique": True}),
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "time_utc": FieldSpec(
            name="time_utc",
            dtype="datetime",
            required=True,
            constraints={"datetime": {"allow_naive": False}},
        ),
        "latitude": FieldSpec(
            name="latitude",
            dtype="float",
            required=True,
            constraints={"range": {"min": -90, "max": 90}},
        ),
        "longitude": FieldSpec(
            name="longitude",
            dtype="float",
            required=True,
            constraints={"range": {"min": -180, "max": 180}},
        ),
        "location_ref": FieldSpec(name="location_ref", dtype="string", required=False),
        "poi_cat": FieldSpec(name="poi_cat", dtype="string", required=False),
        "accuracy": FieldSpec(name="accuracy", dtype="float", required=False),
        "device_type": FieldSpec(name="device_type", dtype="string", required=False),
        "source_app": FieldSpec(name="source_app", dtype="string", required=False),
        "confidence": FieldSpec(name="confidence", dtype="float", required=False),
        "note": FieldSpec(name="note", dtype="string", required=False),
        "provider": FieldSpec(name="provider", dtype="string", required=False),
    }
    return TraceSchema(
        version="trace-v1-rich",
        fields=fields,
        required=list(TRACE_MIN_FIELDS),
        crs="EPSG:4326",
        timezone="America/Santiago",
    )


def _base_trip_fields() -> dict[str, FieldSpec]:
    return {
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=True),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=True),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float", required=True),
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float", required=True),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=True),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=True),
        "origin_h3_index": FieldSpec(name="origin_h3_index", dtype="string", required=True),
        "destination_h3_index": FieldSpec(name="destination_h3_index", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=True),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=True),
    }


def make_trip_schema_min() -> TripSchema:
    fields = _base_trip_fields()
    return TripSchema(
        version="trip-v1-min",
        fields=fields,
        required=list(TRIP_MIN_FIELDS),
    )


def make_trip_schema_rich_bootstrap() -> TripSchema:
    fields = _base_trip_fields()
    fields.update(
        {
            "origin_location_ref": FieldSpec(name="origin_location_ref", dtype="string", required=False),
            "destination_location_ref": FieldSpec(name="destination_location_ref", dtype="string", required=False),
            "origin_device_type": FieldSpec(name="origin_device_type", dtype="string", required=False),
            "destination_device_type": FieldSpec(name="destination_device_type", dtype="string", required=False),
            "origin_accuracy": FieldSpec(name="origin_accuracy", dtype="float", required=False),
            "destination_accuracy": FieldSpec(name="destination_accuracy", dtype="float", required=False),
            "origin_poi_cat": FieldSpec(
                name="origin_poi_cat",
                dtype="categorical",
                required=False,
                domain=DomainSpec(values=[], extendable=True),
            ),
            "destination_poi_cat": FieldSpec(
                name="destination_poi_cat",
                dtype="categorical",
                required=False,
                domain=DomainSpec(values=[], extendable=True),
            ),
        }
    )
    return TripSchema(
        version="trip-v1-rich-bootstrap",
        fields=fields,
        required=list(TRIP_MIN_FIELDS),
    )


def make_trip_schema_rich_extendable() -> TripSchema:
    fields = _base_trip_fields()
    fields.update(
        {
            "origin_location_ref": FieldSpec(name="origin_location_ref", dtype="string", required=False),
            "destination_location_ref": FieldSpec(name="destination_location_ref", dtype="string", required=False),
            "origin_poi_cat": FieldSpec(
                name="origin_poi_cat",
                dtype="categorical",
                required=False,
                domain=DomainSpec(values=["home", "work"], extendable=True),
            ),
            "destination_poi_cat": FieldSpec(
                name="destination_poi_cat",
                dtype="categorical",
                required=False,
                domain=DomainSpec(values=["home", "work"], extendable=True),
            ),
        }
    )
    return TripSchema(
        version="trip-v1-rich-extendable",
        fields=fields,
        required=list(TRIP_MIN_FIELDS),
    )


def make_trip_schema_rich_blocked() -> TripSchema:
    fields = _base_trip_fields()
    fields.update(
        {
            "origin_poi_cat": FieldSpec(
                name="origin_poi_cat",
                dtype="categorical",
                required=False,
                domain=DomainSpec(values=["home", "work"], extendable=False),
            ),
            "destination_poi_cat": FieldSpec(
                name="destination_poi_cat",
                dtype="categorical",
                required=False,
                domain=DomainSpec(values=["home", "work"], extendable=False),
            ),
        }
    )
    return TripSchema(
        version="trip-v1-rich-blocked",
        fields=fields,
        required=list(TRIP_MIN_FIELDS),
    )

### 0.5 Fixtures ricas de traces

In [5]:
def make_trace_points_rich_df() -> pd.DataFrame:
    df = pd.DataFrame(
        {
            "point_id": ["p0", "p1", "p2", "p3", "p4", "q0", "q1", "q2", "q3"],
            "user_id": ["u1", "u1", "u1", "u1", "u1", "u2", "u2", "u2", "u2"],
            "time_utc": pd.to_datetime(
                [
                    "2026-03-10T08:00:00Z",
                    "2026-03-10T08:25:00Z",
                    "2026-03-10T09:10:00Z",
                    "2026-03-10T12:30:00Z",
                    "2026-03-10T18:10:00Z",
                    "2026-03-10T07:40:00Z",
                    "2026-03-10T08:15:00Z",
                    "2026-03-10T17:35:00Z",
                    "2026-03-10T19:05:00Z",
                ],
                utc=True,
            ),
            "latitude": [-33.4500, -33.4560, -33.4600, -33.4550, -33.4505, -33.4700, -33.4725, -33.4680, -33.4705],
            "longitude": [-70.6600, -70.6500, -70.6400, -70.6450, -70.6605, -70.6800, -70.6720, -70.6760, -70.6805],
            "location_ref": [
                "home_u1_am",
                "cafe_u1",
                "work_u1",
                "lunch_u1",
                "home_u1_pm",
                "home_u2_am",
                "school_u2",
                "gym_u2",
                "home_u2_pm",
            ],
            "poi_cat": ["home", "food", "work", "food", "home", "home", "education", "leisure", "home"],
            "accuracy": [5.0, 8.0, 7.0, 9.0, 6.0, 6.0, 7.0, 5.0, 6.0],
            "device_type": ["phone", "phone", "phone", "phone", "phone", "watch", "watch", "watch", "watch"],
            "source_app": ["foursquare"] * 5 + ["app_b"] * 4,
            "confidence": [0.95, 0.92, 0.93, 0.91, 0.94, 0.90, 0.89, 0.88, 0.90],
            "note": [
                "u1_start",
                "u1_coffee",
                "u1_work",
                "u1_lunch",
                "u1_return",
                "u2_start",
                "u2_school",
                "u2_gym",
                "u2_return",
            ],
            "provider": ["provider_a"] * 5 + ["provider_b"] * 4,
        }
    )
    return df


def make_trace_clusters_rich_df() -> pd.DataFrame:
    df = pd.DataFrame(
        {
            "point_id": ["p0", "p1", "p2", "p3", "p4", "p5", "q0", "q1", "q2", "q3", "q4", "q5"],
            "user_id": ["u1"] * 6 + ["u2"] * 6,
            "time_utc": pd.to_datetime(
                [
                    "2026-03-11T08:00:00Z",
                    "2026-03-11T08:03:00Z",
                    "2026-03-11T08:40:00Z",
                    "2026-03-11T08:41:00Z",
                    "2026-03-11T09:20:00Z",
                    "2026-03-11T09:23:00Z",
                    "2026-03-11T07:30:00Z",
                    "2026-03-11T07:32:00Z",
                    "2026-03-11T08:15:00Z",
                    "2026-03-11T08:16:00Z",
                    "2026-03-11T18:10:00Z",
                    "2026-03-11T18:12:00Z",
                ],
                utc=True,
            ),
            "latitude": [-33.4500, -33.4501, -33.4550, -33.4551, -33.4600, -33.4601, -33.4700, -33.4701, -33.4725, -33.4726, -33.4703, -33.4704],
            "longitude": [-70.6600, -70.6601, -70.6500, -70.6501, -70.6400, -70.6401, -70.6800, -70.6801, -70.6720, -70.6721, -70.6803, -70.6804],
            "location_ref": [
                "home_u1_a",
                "home_u1_b",
                "cafe_u1_a",
                "cafe_u1_b",
                "work_u1_a",
                "work_u1_b",
                "home_u2_a",
                "home_u2_b",
                "school_u2_a",
                "school_u2_b",
                "home_u2_pm_a",
                "home_u2_pm_b",
            ],
            "poi_cat": ["home", "home", "food", "food", "work", "work", "home", "home", "education", "education", "home", "home"],
            "accuracy": [4.0, 5.0, 6.0, 6.5, 5.5, 5.0, 4.5, 4.8, 6.2, 6.1, 4.9, 5.2],
            "device_type": ["phone"] * 6 + ["watch"] * 6,
            "source_app": ["foursquare"] * 6 + ["app_b"] * 6,
            "confidence": [0.97, 0.96, 0.93, 0.92, 0.95, 0.94, 0.91, 0.90, 0.89, 0.88, 0.92, 0.91],
            "note": [
                "u1_home_a",
                "u1_home_b",
                "u1_cafe_a",
                "u1_cafe_b",
                "u1_work_a",
                "u1_work_b",
                "u2_home_a",
                "u2_home_b",
                "u2_school_a",
                "u2_school_b",
                "u2_home_pm_a",
                "u2_home_pm_b",
            ],
            "provider": ["provider_a"] * 6 + ["provider_b"] * 6,
        }
    )
    return df


def make_trace_dataset(
    df: pd.DataFrame,
    *,
    validated: bool,
    dataset_id: str,
    events: list[dict] | None = None,
) -> TraceDataset:
    if events is None:
        events = [{"op": "import_traces"}, {"op": "validate_traces"}] if validated else []
    return TraceDataset(
        data=df.copy(deep=True),
        schema=make_trace_schema_rich(),
        provenance={"source_name": "synthetic", "fixture": dataset_id},
        metadata={
            "dataset_id": dataset_id,
            "schema_version": "trace-v1-rich",
            "is_validated": validated,
            "events": deepcopy(events),
        },
    )

### 0.6 Fixtures raw para los tests puente de import + validate + infer

In [6]:
RAW_FIELD_MAP_NO_POINT_ID = {
    "user_id": "uid",
    "time_utc": "observed_local",
    "latitude": "lat_src",
    "longitude": "lon_src",
    "location_ref": "venue_id",
    "poi_cat": "venue_cat",
    "accuracy": "accuracy_m",
    "device_type": "device_src",
    "source_app": "source_app_raw",
    "confidence": "confidence_score",
    "note": "note_raw",
    "provider": "provider_name",
}

RAW_FIELD_MAP_WITH_POINT_ID = {
    "point_id": "raw_pid",
    "user_id": "uid",
    "time_utc": "observed_local",
    "latitude": "lat_src",
    "longitude": "lon_src",
    "location_ref": "venue_id",
    "poi_cat": "venue_cat",
    "accuracy": "accuracy_m",
    "device_type": "device_src",
    "source_app": "source_app_raw",
    "confidence": "confidence_score",
    "note": "note_raw",
    "provider": "provider_name",
}


def make_raw_points_no_pointid_df() -> pd.DataFrame:
    base = make_trace_points_rich_df().copy()
    local_naive = base["time_utc"].dt.tz_convert("America/Santiago").dt.strftime("%Y-%m-%d %H:%M:%S")
    return pd.DataFrame(
        {
            "uid": base["user_id"],
            "observed_local": local_naive,
            "lat_src": base["latitude"],
            "lon_src": base["longitude"],
            "venue_id": base["location_ref"],
            "venue_cat": base["poi_cat"],
            "accuracy_m": base["accuracy"],
            "device_src": base["device_type"],
            "source_app_raw": base["source_app"],
            "confidence_score": base["confidence"],
            "note_raw": base["note"],
            "provider_name": base["provider"],
            "raw_batch": ["batch_A"] * len(base),
        }
    )


def make_raw_clusters_with_pointid_df() -> pd.DataFrame:
    base = make_trace_clusters_rich_df().copy()
    local_naive = base["time_utc"].dt.tz_convert("America/Santiago").dt.strftime("%Y-%m-%d %H:%M:%S")
    return pd.DataFrame(
        {
            "raw_pid": base["point_id"],
            "uid": base["user_id"],
            "observed_local": local_naive,
            "lat_src": base["latitude"],
            "lon_src": base["longitude"],
            "venue_id": base["location_ref"],
            "venue_cat": base["poi_cat"],
            "accuracy_m": base["accuracy"],
            "device_src": base["device_type"],
            "source_app_raw": base["source_app"],
            "confidence_score": base["confidence"],
            "note_raw": base["note"],
            "provider_name": base["provider"],
            "raw_batch": ["batch_B"] * len(base),
            "raw_quality_flag": ["ok"] * len(base),
        }
    )

### 0.7 Helpers para options efectivas

In [7]:
def make_points_options(**overrides) -> InferTripsOptions:
    payload = dict(
        infer_mode="consecutive_points",
        strict=False,
        strict_domains=False,
        require_validated_traces=True,
        drop_invalid=True,
        h3_resolution=8,
        max_time_delta_s=None,
        min_time_delta_s=None,
        min_distance_m=None,
        cluster_radius_m=None,
        cluster_max_time_gap_s=None,
        propagate_trace_fields=None,
    )
    payload.update(overrides)
    return InferTripsOptions(**payload)


def make_cluster_options(**overrides) -> InferTripsOptions:
    payload = dict(
        infer_mode="consecutive_clusters",
        strict=False,
        strict_domains=False,
        require_validated_traces=True,
        drop_invalid=True,
        h3_resolution=8,
        max_time_delta_s=None,
        min_time_delta_s=None,
        min_distance_m=None,
        cluster_radius_m=50.0,
        cluster_max_time_gap_s=300.0,
        propagate_trace_fields=None,
    )
    payload.update(overrides)
    return InferTripsOptions(**payload)

### 0.8 Configuración de display

In [8]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

## Sección 1. Integration tests de `infer_trips_from_traces()`

### Test 1 - caso feliz directo en `consecutive_points` con fixture rico y propagación explícita

Se prueba el camino principal sobre un `TraceDataset` ya validado, con varias filas y columnas extra.  
Aquí interesa verificar simultáneamente:

- tabla de salida;
- `InferenceReport`;
- derivación H3;
- `metadata/evento`;
- `provenance`;
- `schema_effective`;
- y que el input no se muta.

In [9]:
traces = make_trace_dataset(
    make_trace_points_rich_df(),
    validated=True,
    dataset_id="trace_points_validated_rich",
)
traces_before_df = traces.data.copy(deep=True)
traces_before_meta = deepcopy(traces.metadata)

trip_schema = make_trip_schema_rich_bootstrap()
options = make_points_options(
    propagate_trace_fields={
        "location_ref": "both",
        "poi_cat": "both",
        "device_type": "origin",
        "accuracy": "destination",
    }
)

trip_dataset, report = infer_trips_from_traces(
    traces,
    trip_schema,
    options=options,
    value_correspondence={
        "origin_poi_cat": {"education": "study", "leisure": "wellbeing"},
        "destination_poi_cat": {"education": "study", "leisure": "wellbeing"},
    },
    provenance={"notebook": "integration_tests_op16", "case": "test_1_points_happy_path"},
)

print("Dataset de traces, trips y las issues del infer_trips")
display(traces.data)
display(trip_dataset.data)
display(report.issues)

assert isinstance(trip_dataset, TripDataset)
assert isinstance(report, InferenceReport)

# 5 puntos u1 -> 4 viajes, 4 puntos u2 -> 3 viajes
assert len(trip_dataset.data) == 7
assert report.summary["infer_mode"] == "consecutive_points"
assert report.summary["n_points_in"] == len(traces_before_df)
assert report.summary["n_trips_out"] == len(trip_dataset.data)

assert "origin_h3_index" in trip_dataset.data.columns
assert "destination_h3_index" in trip_dataset.data.columns
assert "origin_location_ref" in trip_dataset.data.columns
assert "destination_location_ref" in trip_dataset.data.columns
assert "origin_device_type" in trip_dataset.data.columns
assert "destination_accuracy" in trip_dataset.data.columns

assert trip_dataset.field_correspondence == {}
assert trip_dataset.metadata["is_validated"] is False
assert trip_dataset.metadata["events"][0]["op"] == "infer_trips"
assert trip_dataset.metadata["events"][0]["parameters"]["infer_mode"] == "consecutive_points"
assert trip_dataset.metadata["h3"]["resolution"] == 8
assert "field_propagation" in trip_dataset.metadata["mappings"]
assert trip_dataset.provenance["prior_events_summary"]["n_events"] == 2
assert trip_dataset.provenance["prior_events_summary"]["last_event_op"] == "validate_traces"
assert trip_dataset.schema_effective.temporal["tier"] == "tier_1"

assert_has_code(report, "INF.H3.DERIVED")
assert_has_code(report, "INF.OK.SUMMARY")

# Verificación de no mutación del input
assert_frame_equal(traces.data, traces_before_df)
assert traces.metadata == traces_before_meta

show_ok("Test 1 - consecutive_points rico")

Dataset de traces, trips y las issues del infer_trips


,point_id,user_id,time_utc,latitude,longitude,location_ref,poi_cat,accuracy,device_type,source_app,confidence,note,provider
0,p0,u1,2026-03-10 08:00:00+00:00,-33.4500,-70.6600,home_u1_am,home,5.0,phone,foursquare,0.95,u1_start,provider_a
1,p1,u1,2026-03-10 08:25:00+00:00,-33.4560,-70.6500,cafe_u1,food,8.0,phone,foursquare,0.92,u1_coffee,provider_a
2,p2,u1,2026-03-10 09:10:00+00:00,-33.4600,-70.6400,work_u1,work,7.0,phone,foursquare,0.93,u1_work,provider_a
3,p3,u1,2026-03-10 12:30:00+00:00,-33.4550,-70.6450,lunch_u1,food,9.0,phone,foursquare,0.91,u1_lunch,provider_a
4,p4,u1,2026-03-10 18:10:00+00:00,-33.4505,-70.6605,home_u1_pm,home,6.0,phone,foursquare,0.94,u1_return,provider_a
5,q0,u2,2026-03-10 07:40:00+00:00,-33.4700,-70.6800,home_u2_am,home,6.0,watch,app_b,0.90,u2_start,provider_b
6,q1,u2,2026-03-10 08:15:00+00:00,-33.4725,-70.6720,school_u2,education,7.0,watch,app_b,0.89,u2_school,provider_b
7,q2,u2,2026-03-10 17:35:00+00:00,-33.4680,-70.6760,gym_u2,leisure,5.0,watch,app_b,0.88,u2_gym,provider_b
8,q3,u2,2026-03-10 19:05:00+00:00,-33.4705,-70.6805,home_u2_pm,home,6.0,watch,app_b,0.90,u2_return,provider_b


,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,trip_id,movement_seq,origin_location_ref,destination_location_ref,origin_poi_cat,destination_poi_cat,origin_device_type,destination_accuracy,origin_h3_index,destination_h3_index
0,m0,u1,-70.660,-33.4500,-70.6500,-33.4560,2026-03-10 08:00:00+00:00,2026-03-10 08:25:00+00:00,m0,0,home_u1_am,cafe_u1,home,food,phone,8.0,88b2c55417fffff,88b2c554ebfffff
1,m1,u1,-70.650,-33.4560,-70.6400,-33.4600,2026-03-10 08:25:00+00:00,2026-03-10 09:10:00+00:00,m1,0,cafe_u1,work_u1,food,work,phone,7.0,88b2c554ebfffff,88b2c5548dfffff
2,m2,u1,-70.640,-33.4600,-70.6450,-33.4550,2026-03-10 09:10:00+00:00,2026-03-10 12:30:00+00:00,m2,0,work_u1,lunch_u1,work,food,phone,9.0,88b2c5548dfffff,88b2c554ebfffff
3,m3,u1,-70.645,-33.4550,-70.6605,-33.4505,2026-03-10 12:30:00+00:00,2026-03-10 18:10:00+00:00,m3,0,lunch_u1,home_u1_pm,food,home,phone,6.0,88b2c554ebfffff,88b2c554edfffff
4,m4,u2,-70.680,-33.4700,-70.6720,-33.4725,2026-03-10 07:40:00+00:00,2026-03-10 08:15:00+00:00,m4,0,home_u2_am,school_u2,home,study,watch,7.0,88b2c555d1fffff,88b2c555d3fffff
5,m5,u2,-70.672,-33.4725,-70.6760,-33.4680,2026-03-10 08:15:00+00:00,2026-03-10 17:35:00+00:00,m5,0,school_u2,gym_u2,study,wellbeing,watch,5.0,88b2c555d3fffff,88b2c555d1fffff
6,m6,u2,-70.676,-33.4680,-70.6805,-33.4705,2026-03-10 17:35:00+00:00,2026-03-10 19:05:00+00:00,m6,0,gym_u2,home_u2_pm,wellbeing,home,watch,6.0,88b2c555d1fffff,88b2c555d1fffff


[Issue(level='info', code='INF.CANDIDATES.POINTS_MODE_APPLIED', message='Se construyeron 7 candidatos OD a partir de pares consecutivos por usuario.', field=None, source_field=None, row_count=None, details={'infer_mode': 'consecutive_points', 'strict': False, 'strict_domains': False, 'drop_invalid': True, 'require_validated_traces': True, 'h3_resolution': 8, 'max_time_delta_s': None, 'min_time_delta_s': None, 'min_distance_m': None, 'cluster_radius_m': None, 'cluster_max_time_gap_s': None, 'n_points_in': 9, 'n_users_in': 2, 'n_candidates_in': 7, 'action': 'built_point_candidates'}),
 Issue(level='info', code='INF.PROPAGATION.APPLIED', message='Se propagaron campos extra desde traces hacia el output de trips (nuevas columnas=6).', field=None, source_field=None, row_count=None, details={'infer_mode': 'consecutive_points', 'strict': False, 'strict_domains': False, 'drop_invalid': True, 'require_validated_traces': True, 'h3_resolution': 8, 'max_time_delta_s': None, 'min_time_delta_s': None

OK - Test 1 - consecutive_points rico


### Test 2 - caso feliz directo en `consecutive_clusters` con chequeo de puntos frontera

Aquí se prueba el modo clusters sobre bursts ricos.  
Además de validar el resultado general, se verifica algo importante del contrato: en el viaje entre clusters consecutivos, el origen debe salir del último punto del cluster origen y el destino del primer punto del cluster destino.

In [10]:
traces = make_trace_dataset(
    make_trace_clusters_rich_df(),
    validated=True,
    dataset_id="trace_clusters_validated_rich",
)

trip_schema = make_trip_schema_rich_bootstrap()
options = make_cluster_options(
    cluster_radius_m=50.0,
    cluster_max_time_gap_s=300.0,
    propagate_trace_fields={
        "location_ref": "both",
        "poi_cat": "both",
        "accuracy": "both",
    },
)

trip_dataset, report = infer_trips_from_traces(
    traces,
    trip_schema,
    options=options,
    value_correspondence=None,
    provenance={"notebook": "integration_tests_op16", "case": "test_2_clusters_happy_path"},
)

print("Dataset de traces, trips y las issues del infer_trips")
display(traces.data)
display(trip_dataset.data)
display(report.issues)

assert isinstance(trip_dataset, TripDataset)
assert isinstance(report, InferenceReport)

# u1: 3 clusters -> 2 viajes; u2: 3 clusters -> 2 viajes
assert len(trip_dataset.data) == 4
assert report.summary["infer_mode"] == "consecutive_clusters"
assert report.summary["n_clusters_out"] == 6
assert report.summary["n_trips_out"] == 4

# Primer viaje de u1 debe usar último punto del cluster home y primer punto del cluster cafe
u1_first_trip = trip_dataset.data.loc[trip_dataset.data["user_id"] == "u1"].iloc[0]
assert u1_first_trip["origin_location_ref"] == "home_u1_b"
assert u1_first_trip["destination_location_ref"] == "cafe_u1_a"
assert pd.Timestamp(u1_first_trip["origin_time_utc"]) == pd.Timestamp("2026-03-11T08:03:00Z")
assert pd.Timestamp(u1_first_trip["destination_time_utc"]) == pd.Timestamp("2026-03-11T08:40:00Z")

assert_has_code(report, "INF.CLUSTERS.MODE_APPLIED")
assert_has_code(report, "INF.H3.DERIVED")
assert_has_code(report, "INF.OK.SUMMARY")

show_ok("Test 2 - consecutive_clusters rico con puntos frontera")

Dataset de traces, trips y las issues del infer_trips


,point_id,user_id,time_utc,latitude,longitude,location_ref,poi_cat,accuracy,device_type,source_app,confidence,note,provider
0,p0,u1,2026-03-11 08:00:00+00:00,-33.4500,-70.6600,home_u1_a,home,4.0,phone,foursquare,0.97,u1_home_a,provider_a
1,p1,u1,2026-03-11 08:03:00+00:00,-33.4501,-70.6601,home_u1_b,home,5.0,phone,foursquare,0.96,u1_home_b,provider_a
2,p2,u1,2026-03-11 08:40:00+00:00,-33.4550,-70.6500,cafe_u1_a,food,6.0,phone,foursquare,0.93,u1_cafe_a,provider_a
3,p3,u1,2026-03-11 08:41:00+00:00,-33.4551,-70.6501,cafe_u1_b,food,6.5,phone,foursquare,0.92,u1_cafe_b,provider_a
4,p4,u1,2026-03-11 09:20:00+00:00,-33.4600,-70.6400,work_u1_a,work,5.5,phone,foursquare,0.95,u1_work_a,provider_a
5,p5,u1,2026-03-11 09:23:00+00:00,-33.4601,-70.6401,work_u1_b,work,5.0,phone,foursquare,0.94,u1_work_b,provider_a
6,q0,u2,2026-03-11 07:30:00+00:00,-33.4700,-70.6800,home_u2_a,home,4.5,watch,app_b,0.91,u2_home_a,provider_b
7,q1,u2,2026-03-11 07:32:00+00:00,-33.4701,-70.6801,home_u2_b,home,4.8,watch,app_b,0.90,u2_home_b,provider_b
8,q2,u2,2026-03-11 08:15:00+00:00,-33.4725,-70.6720,school_u2_a,education,6.2,watch,app_b,0.89,u2_school_a,provider_b
9,q3,u2,2026-03-11 08:16:00+00:00,-33.4726,-70.6721,school_u2_b,education,6.1,watch,app_b,0.88,u2_school_b,provider_b


,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,trip_id,movement_seq,origin_location_ref,destination_location_ref,origin_poi_cat,destination_poi_cat,origin_accuracy,destination_accuracy,origin_h3_index,destination_h3_index
0,m0,u1,-70.6601,-33.4501,-70.6500,-33.4550,2026-03-11 08:03:00+00:00,2026-03-11 08:40:00+00:00,m0,0,home_u1_b,cafe_u1_a,home,food,5.0,6.0,88b2c554edfffff,88b2c554ebfffff
1,m1,u1,-70.6501,-33.4551,-70.6400,-33.4600,2026-03-11 08:41:00+00:00,2026-03-11 09:20:00+00:00,m1,0,cafe_u1_b,work_u1_a,food,work,6.5,5.5,88b2c554ebfffff,88b2c5548dfffff
2,m2,u2,-70.6801,-33.4701,-70.6720,-33.4725,2026-03-11 07:32:00+00:00,2026-03-11 08:15:00+00:00,m2,0,home_u2_b,school_u2_a,home,education,4.8,6.2,88b2c555d1fffff,88b2c555d3fffff
3,m3,u2,-70.6721,-33.4726,-70.6803,-33.4703,2026-03-11 08:16:00+00:00,2026-03-11 18:10:00+00:00,m3,0,school_u2_b,home_u2_pm_a,education,home,6.1,4.9,88b2c555d3fffff,88b2c555d1fffff


[Issue(level='info', code='INF.CLUSTERS.MODE_APPLIED', message='Se construyeron 6 clusters secuenciales y 4 candidatos OD entre clusters consecutivos.', field=None, source_field=None, row_count=None, details={'infer_mode': 'consecutive_clusters', 'strict': False, 'strict_domains': False, 'drop_invalid': True, 'require_validated_traces': True, 'h3_resolution': 8, 'max_time_delta_s': None, 'min_time_delta_s': None, 'min_distance_m': None, 'cluster_radius_m': 50.0, 'cluster_max_time_gap_s': 300.0, 'n_points_in': 12, 'n_users_in': 2, 'n_clusters_out': 6, 'clusters_sample': [{'cluster_id': 0, 'user_id': 'u1', 'cluster_start_utc': '2026-03-11T08:00:00Z', 'cluster_end_utc': '2026-03-11T08:03:00Z', 'first_point_id': 'p0', 'last_point_id': 'p1', 'n_points': 2, 'first_row_idx': 0, 'last_row_idx': 1}, {'cluster_id': 1, 'user_id': 'u1', 'cluster_start_utc': '2026-03-11T08:40:00Z', 'cluster_end_utc': '2026-03-11T08:41:00Z', 'first_point_id': 'p2', 'last_point_id': 'p3', 'n_points': 2, 'first_row_id

OK - Test 2 - consecutive_clusters rico con puntos frontera


### Test 3 - puente real `import_traces -> validate_traces -> infer` en modo points, con `point_id` generado

Este caso prueba una cadena real del bloque traces antes de OP-16.  
Se usa un dataframe crudo con varias columnas, sin `point_id`, de modo que OP-14 tenga que generarlo, OP-15 lo valide, y OP-16 lo consuma ya validado.

In [12]:
raw_df = make_raw_points_no_pointid_df()

trace_schema = make_trace_schema_rich()
trip_schema = make_trip_schema_min()

traces_imported, import_report = import_traces_from_dataframe(
    raw_df,
    trace_schema,
    source_name="raw_points_no_pointid",
    options=ImportTraceOptions(
        keep_extra_fields=True,
        selected_fields=None,
        strict=False,
        source_timezone="America/Santiago",
    ),
    field_correspondence=RAW_FIELD_MAP_NO_POINT_ID,
    provenance={"case": "test_3_import_validate_infer_points"},
)

assert import_report.summary["rows_in"] == len(raw_df)
assert import_report.summary["point_id_generated"] is True
assert traces_imported.metadata["is_validated"] is False
assert traces_imported.metadata["events"][0]["op"] == "import_traces"
assert "point_id" in traces_imported.data.columns
assert "location_ref" in traces_imported.data.columns
assert "poi_cat" in traces_imported.data.columns
assert "device_type" in traces_imported.data.columns
assert "raw_batch" in traces_imported.data.columns  # keep_extra_fields=True

val_report = validate_traces(
    traces_imported,
    options=TraceValidationOptions(strict=False),
)
print("Dataset de traces raw, dataset importado, issues de import traces y report de validate")
display(raw_df)
display(traces_imported.data)
display(import_report.issues)
display(val_report)

assert val_report.summary["ok"] is True
assert traces_imported.metadata["is_validated"] is True
assert traces_imported.metadata["events"][-1]["op"] == "validate_traces"

trip_dataset, infer_report = infer_trips_from_traces(
    traces_imported,
    trip_schema,
    options=make_points_options(),
    value_correspondence=None,
    provenance={"case": "test_3_import_validate_infer_points"},
)

print("trips inferidos y las issues del infer_trips")
display(trip_dataset.data)
display(infer_report.issues)

assert infer_report.summary["infer_mode"] == "consecutive_points"
assert infer_report.summary["n_points_in"] == len(traces_imported.data)
assert len(trip_dataset.data) == 7

# El output no copia el historial completo de events de traces; lo resume en provenance.
assert len(trip_dataset.metadata["events"]) == 1
assert trip_dataset.metadata["events"][0]["op"] == "infer_trips"
assert trip_dataset.provenance["prior_events_summary"]["n_events"] == 2
assert trip_dataset.provenance["prior_events_summary"]["ops"] == ["import_traces", "validate_traces"]

show_ok("Test 3 - puente import -> validate -> infer (points)")

Dataset de traces raw, dataset importado, issues de import traces y report de validate


,uid,observed_local,lat_src,lon_src,venue_id,venue_cat,accuracy_m,device_src,source_app_raw,confidence_score,note_raw,provider_name,raw_batch
0,u1,2026-03-10 05:00:00,-33.4500,-70.6600,home_u1_am,home,5.0,phone,foursquare,0.95,u1_start,provider_a,batch_A
1,u1,2026-03-10 05:25:00,-33.4560,-70.6500,cafe_u1,food,8.0,phone,foursquare,0.92,u1_coffee,provider_a,batch_A
2,u1,2026-03-10 06:10:00,-33.4600,-70.6400,work_u1,work,7.0,phone,foursquare,0.93,u1_work,provider_a,batch_A
3,u1,2026-03-10 09:30:00,-33.4550,-70.6450,lunch_u1,food,9.0,phone,foursquare,0.91,u1_lunch,provider_a,batch_A
4,u1,2026-03-10 15:10:00,-33.4505,-70.6605,home_u1_pm,home,6.0,phone,foursquare,0.94,u1_return,provider_a,batch_A
5,u2,2026-03-10 04:40:00,-33.4700,-70.6800,home_u2_am,home,6.0,watch,app_b,0.90,u2_start,provider_b,batch_A
6,u2,2026-03-10 05:15:00,-33.4725,-70.6720,school_u2,education,7.0,watch,app_b,0.89,u2_school,provider_b,batch_A
7,u2,2026-03-10 14:35:00,-33.4680,-70.6760,gym_u2,leisure,5.0,watch,app_b,0.88,u2_gym,provider_b,batch_A
8,u2,2026-03-10 16:05:00,-33.4705,-70.6805,home_u2_pm,home,6.0,watch,app_b,0.90,u2_return,provider_b,batch_A


,point_id,user_id,time_utc,latitude,longitude,location_ref,poi_cat,accuracy,device_type,source_app,confidence,note,provider,raw_batch
0,p0,u1,2026-03-10 08:00:00+00:00,-33.4500,-70.6600,home_u1_am,home,5.0,phone,foursquare,0.95,u1_start,provider_a,batch_A
1,p1,u1,2026-03-10 08:25:00+00:00,-33.4560,-70.6500,cafe_u1,food,8.0,phone,foursquare,0.92,u1_coffee,provider_a,batch_A
2,p2,u1,2026-03-10 09:10:00+00:00,-33.4600,-70.6400,work_u1,work,7.0,phone,foursquare,0.93,u1_work,provider_a,batch_A
3,p3,u1,2026-03-10 12:30:00+00:00,-33.4550,-70.6450,lunch_u1,food,9.0,phone,foursquare,0.91,u1_lunch,provider_a,batch_A
4,p4,u1,2026-03-10 18:10:00+00:00,-33.4505,-70.6605,home_u1_pm,home,6.0,phone,foursquare,0.94,u1_return,provider_a,batch_A
5,p5,u2,2026-03-10 07:40:00+00:00,-33.4700,-70.6800,home_u2_am,home,6.0,watch,app_b,0.90,u2_start,provider_b,batch_A
6,p6,u2,2026-03-10 08:15:00+00:00,-33.4725,-70.6720,school_u2,education,7.0,watch,app_b,0.89,u2_school,provider_b,batch_A
7,p7,u2,2026-03-10 17:35:00+00:00,-33.4680,-70.6760,gym_u2,leisure,5.0,watch,app_b,0.88,u2_gym,provider_b,batch_A
8,p8,u2,2026-03-10 19:05:00+00:00,-33.4705,-70.6805,home_u2_pm,home,6.0,watch,app_b,0.90,u2_return,provider_b,batch_A


[Issue(level='info', code='IMP.CORE.POINT_ID_GENERATED', message='No se encontró point_id alcanzable; se generó automáticamente una columna técnica secuencial.', field='point_id', source_field=None, row_count=None, details={'field': 'point_id', 'insert_position': 0, 'rows_out': 9, 'action': 'generate_point_id'})]

ConsistencyReport(issues=[], summary={'ok': True, 'n_rows': 9, 'n_issues': 0, 'n_errors': 0, 'n_warnings': 0, 'n_info': 0, 'counts_by_level': {'error': 0, 'warning': 0, 'info': 0}, 'counts_by_code': {}, 'checked_fields': ['point_id', 'user_id', 'time_utc', 'latitude', 'longitude', 'location_ref', 'poi_cat', 'accuracy', 'device_type', 'source_app', 'confidence', 'note', 'provider'], 'checks_executed': {'required_fields': True, 'types_and_formats': True, 'constraints': True, 'monotonic_time_per_user': True}, 'schema_version': 'trace-v1-rich'}, parameters={})

trips inferidos y las issues del infer_trips


,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,trip_id,movement_seq,origin_h3_index,destination_h3_index
0,m0,u1,-70.660,-33.4500,-70.6500,-33.4560,2026-03-10 08:00:00+00:00,2026-03-10 08:25:00+00:00,m0,0,88b2c55417fffff,88b2c554ebfffff
1,m1,u1,-70.650,-33.4560,-70.6400,-33.4600,2026-03-10 08:25:00+00:00,2026-03-10 09:10:00+00:00,m1,0,88b2c554ebfffff,88b2c5548dfffff
2,m2,u1,-70.640,-33.4600,-70.6450,-33.4550,2026-03-10 09:10:00+00:00,2026-03-10 12:30:00+00:00,m2,0,88b2c5548dfffff,88b2c554ebfffff
3,m3,u1,-70.645,-33.4550,-70.6605,-33.4505,2026-03-10 12:30:00+00:00,2026-03-10 18:10:00+00:00,m3,0,88b2c554ebfffff,88b2c554edfffff
4,m4,u2,-70.680,-33.4700,-70.6720,-33.4725,2026-03-10 07:40:00+00:00,2026-03-10 08:15:00+00:00,m4,0,88b2c555d1fffff,88b2c555d3fffff
5,m5,u2,-70.672,-33.4725,-70.6760,-33.4680,2026-03-10 08:15:00+00:00,2026-03-10 17:35:00+00:00,m5,0,88b2c555d3fffff,88b2c555d1fffff
6,m6,u2,-70.676,-33.4680,-70.6805,-33.4705,2026-03-10 17:35:00+00:00,2026-03-10 19:05:00+00:00,m6,0,88b2c555d1fffff,88b2c555d1fffff


[Issue(level='info', code='INF.CANDIDATES.POINTS_MODE_APPLIED', message='Se construyeron 7 candidatos OD a partir de pares consecutivos por usuario.', field=None, source_field=None, row_count=None, details={'infer_mode': 'consecutive_points', 'strict': False, 'strict_domains': False, 'drop_invalid': True, 'require_validated_traces': True, 'h3_resolution': 8, 'max_time_delta_s': None, 'min_time_delta_s': None, 'min_distance_m': None, 'cluster_radius_m': None, 'cluster_max_time_gap_s': None, 'n_points_in': 9, 'n_users_in': 2, 'n_candidates_in': 7, 'action': 'built_point_candidates'}),
 Issue(level='info', code='INF.H3.DERIVED', message='Se derivaron origin_h3_index y destination_h3_index con h3_resolution=8.', field=None, source_field=None, row_count=None, details={'infer_mode': 'consecutive_points', 'strict': False, 'strict_domains': False, 'drop_invalid': True, 'require_validated_traces': True, 'h3_resolution': 8, 'max_time_delta_s': None, 'min_time_delta_s': None, 'min_distance_m': No

OK - Test 3 - puente import -> validate -> infer (points)


### Test 4 - segundo puente real en modo clusters, con `point_id` preservado y `selected_fields` estrictos

Aquí se prueba otra configuración del import de traces:
- el `point_id` ya viene en la fuente;
- `keep_extra_fields=False`;
- `selected_fields` acota qué extras realmente pasan al `TraceDataset`;
- y luego eso se refleja en OP-16 porque solo pueden propagarse los campos que realmente sobrevivieron al import.

In [13]:
raw_df = make_raw_clusters_with_pointid_df()

trace_schema = make_trace_schema_rich()
trip_schema = make_trip_schema_rich_bootstrap()

traces_imported, import_report = import_traces_from_dataframe(
    raw_df,
    trace_schema,
    source_name="raw_clusters_with_pointid",
    options=ImportTraceOptions(
        keep_extra_fields=False,
        selected_fields=["location_ref", "poi_cat", "accuracy", "device_type"],
        strict=False,
        source_timezone="America/Santiago",
    ),
    field_correspondence=RAW_FIELD_MAP_WITH_POINT_ID,
    provenance={"case": "test_4_import_validate_infer_clusters"},
)

assert import_report.summary["point_id_generated"] is False
assert traces_imported.metadata["is_validated"] is False

# Deben sobrevivir solo el núcleo + selected_fields; extras crudos no seleccionados quedan fuera.
assert "location_ref" in traces_imported.data.columns
assert "poi_cat" in traces_imported.data.columns
assert "accuracy" in traces_imported.data.columns
assert "device_type" in traces_imported.data.columns
assert "source_app" not in traces_imported.data.columns
assert "confidence" not in traces_imported.data.columns
assert "note" not in traces_imported.data.columns
assert "provider" not in traces_imported.data.columns
assert "raw_batch" not in traces_imported.data.columns

val_report = validate_traces(traces_imported)
assert val_report.summary["ok"] is True
assert traces_imported.metadata["is_validated"] is True

print("Dataset de traces raw, dataset importado, issues de import traces y report de validate")
display(raw_df)
display(traces_imported.data)
display(import_report.issues)
display(val_report)

trip_dataset, infer_report = infer_trips_from_traces(
    traces_imported,
    trip_schema,
    options=make_cluster_options(
        cluster_radius_m=50.0,
        cluster_max_time_gap_s=300.0,
        propagate_trace_fields={"location_ref": "both", "poi_cat": "both", "accuracy": "origin"},
    ),
    value_correspondence={"origin_poi_cat": {}, "destination_poi_cat": {}},
    provenance={"case": "test_4_import_validate_infer_clusters"},
)

print("trips inferidos y las issues del infer_trips")
display(trip_dataset.data)
display(infer_report.issues)

assert infer_report.summary["infer_mode"] == "consecutive_clusters"
assert infer_report.summary["n_clusters_out"] == 6
assert len(trip_dataset.data) == 4

assert "origin_location_ref" in trip_dataset.data.columns
assert "destination_location_ref" in trip_dataset.data.columns
assert "origin_accuracy" in trip_dataset.data.columns
assert "origin_poi_cat" in trip_dataset.data.columns
assert "destination_poi_cat" in trip_dataset.data.columns

# source_app no sobrevivió al import, por lo tanto tampoco debe aparecer propagado.
assert "origin_source_app" not in trip_dataset.data.columns
assert "destination_source_app" not in trip_dataset.data.columns

show_ok("Test 4 - puente import -> validate -> infer (clusters, selected_fields)")

Dataset de traces raw, dataset importado, issues de import traces y report de validate


,raw_pid,uid,observed_local,lat_src,lon_src,venue_id,venue_cat,accuracy_m,device_src,source_app_raw,confidence_score,note_raw,provider_name,raw_batch,raw_quality_flag
0,p0,u1,2026-03-11 05:00:00,-33.4500,-70.6600,home_u1_a,home,4.0,phone,foursquare,0.97,u1_home_a,provider_a,batch_B,ok
1,p1,u1,2026-03-11 05:03:00,-33.4501,-70.6601,home_u1_b,home,5.0,phone,foursquare,0.96,u1_home_b,provider_a,batch_B,ok
2,p2,u1,2026-03-11 05:40:00,-33.4550,-70.6500,cafe_u1_a,food,6.0,phone,foursquare,0.93,u1_cafe_a,provider_a,batch_B,ok
3,p3,u1,2026-03-11 05:41:00,-33.4551,-70.6501,cafe_u1_b,food,6.5,phone,foursquare,0.92,u1_cafe_b,provider_a,batch_B,ok
4,p4,u1,2026-03-11 06:20:00,-33.4600,-70.6400,work_u1_a,work,5.5,phone,foursquare,0.95,u1_work_a,provider_a,batch_B,ok
5,p5,u1,2026-03-11 06:23:00,-33.4601,-70.6401,work_u1_b,work,5.0,phone,foursquare,0.94,u1_work_b,provider_a,batch_B,ok
6,q0,u2,2026-03-11 04:30:00,-33.4700,-70.6800,home_u2_a,home,4.5,watch,app_b,0.91,u2_home_a,provider_b,batch_B,ok
7,q1,u2,2026-03-11 04:32:00,-33.4701,-70.6801,home_u2_b,home,4.8,watch,app_b,0.90,u2_home_b,provider_b,batch_B,ok
8,q2,u2,2026-03-11 05:15:00,-33.4725,-70.6720,school_u2_a,education,6.2,watch,app_b,0.89,u2_school_a,provider_b,batch_B,ok
9,q3,u2,2026-03-11 05:16:00,-33.4726,-70.6721,school_u2_b,education,6.1,watch,app_b,0.88,u2_school_b,provider_b,batch_B,ok


,point_id,user_id,time_utc,latitude,longitude,location_ref,poi_cat,accuracy,device_type
0,p0,u1,2026-03-11 08:00:00+00:00,-33.4500,-70.6600,home_u1_a,home,4.0,phone
1,p1,u1,2026-03-11 08:03:00+00:00,-33.4501,-70.6601,home_u1_b,home,5.0,phone
2,p2,u1,2026-03-11 08:40:00+00:00,-33.4550,-70.6500,cafe_u1_a,food,6.0,phone
3,p3,u1,2026-03-11 08:41:00+00:00,-33.4551,-70.6501,cafe_u1_b,food,6.5,phone
4,p4,u1,2026-03-11 09:20:00+00:00,-33.4600,-70.6400,work_u1_a,work,5.5,phone
5,p5,u1,2026-03-11 09:23:00+00:00,-33.4601,-70.6401,work_u1_b,work,5.0,phone
6,q0,u2,2026-03-11 07:30:00+00:00,-33.4700,-70.6800,home_u2_a,home,4.5,watch
7,q1,u2,2026-03-11 07:32:00+00:00,-33.4701,-70.6801,home_u2_b,home,4.8,watch
8,q2,u2,2026-03-11 08:15:00+00:00,-33.4725,-70.6720,school_u2_a,education,6.2,watch
9,q3,u2,2026-03-11 08:16:00+00:00,-33.4726,-70.6721,school_u2_b,education,6.1,watch


[Issue(level='info', code='IMP.OPTIONS.EXTRA_FIELDS_DROPPED', message='Se descartaron columnas extra porque la política efectiva de selección/conservación no las admite (n=6).', field=None, source_field=None, row_count=None, details={'keep_extra_fields': False, 'selected_fields': ['location_ref', 'poi_cat', 'accuracy', 'device_type'], 'n_dropped': 6, 'dropped_columns_sample': ['source_app', 'confidence', 'note', 'provider', 'raw_batch', 'raw_quality_flag'], 'dropped_columns_total': 6, 'action': 'drop_extra_fields'})]

ConsistencyReport(issues=[], summary={'ok': True, 'n_rows': 12, 'n_issues': 0, 'n_errors': 0, 'n_warnings': 0, 'n_info': 0, 'counts_by_level': {'error': 0, 'warning': 0, 'info': 0}, 'counts_by_code': {}, 'checked_fields': ['point_id', 'user_id', 'time_utc', 'latitude', 'longitude', 'location_ref', 'poi_cat', 'accuracy', 'device_type'], 'checks_executed': {'required_fields': True, 'types_and_formats': True, 'constraints': True, 'monotonic_time_per_user': True}, 'schema_version': 'trace-v1-rich'}, parameters={})

trips inferidos y las issues del infer_trips


,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,trip_id,movement_seq,origin_location_ref,destination_location_ref,origin_poi_cat,destination_poi_cat,origin_accuracy,origin_h3_index,destination_h3_index
0,m0,u1,-70.6601,-33.4501,-70.6500,-33.4550,2026-03-11 08:03:00+00:00,2026-03-11 08:40:00+00:00,m0,0,home_u1_b,cafe_u1_a,home,food,5.0,88b2c554edfffff,88b2c554ebfffff
1,m1,u1,-70.6501,-33.4551,-70.6400,-33.4600,2026-03-11 08:41:00+00:00,2026-03-11 09:20:00+00:00,m1,0,cafe_u1_b,work_u1_a,food,work,6.5,88b2c554ebfffff,88b2c5548dfffff
2,m2,u2,-70.6801,-33.4701,-70.6720,-33.4725,2026-03-11 07:32:00+00:00,2026-03-11 08:15:00+00:00,m2,0,home_u2_b,school_u2_a,home,education,4.8,88b2c555d1fffff,88b2c555d3fffff
3,m3,u2,-70.6721,-33.4726,-70.6803,-33.4703,2026-03-11 08:16:00+00:00,2026-03-11 18:10:00+00:00,m3,0,school_u2_b,home_u2_pm_a,education,home,6.1,88b2c555d3fffff,88b2c555d1fffff


[Issue(level='info', code='INF.CLUSTERS.MODE_APPLIED', message='Se construyeron 6 clusters secuenciales y 4 candidatos OD entre clusters consecutivos.', field=None, source_field=None, row_count=None, details={'infer_mode': 'consecutive_clusters', 'strict': False, 'strict_domains': False, 'drop_invalid': True, 'require_validated_traces': True, 'h3_resolution': 8, 'max_time_delta_s': None, 'min_time_delta_s': None, 'min_distance_m': None, 'cluster_radius_m': 50.0, 'cluster_max_time_gap_s': 300.0, 'n_points_in': 12, 'n_users_in': 2, 'n_clusters_out': 6, 'clusters_sample': [{'cluster_id': 0, 'user_id': 'u1', 'cluster_start_utc': '2026-03-11T08:00:00Z', 'cluster_end_utc': '2026-03-11T08:03:00Z', 'first_point_id': 'p0', 'last_point_id': 'p1', 'n_points': 2, 'first_row_idx': 0, 'last_row_idx': 1}, {'cluster_id': 1, 'user_id': 'u1', 'cluster_start_utc': '2026-03-11T08:40:00Z', 'cluster_end_utc': '2026-03-11T08:41:00Z', 'first_point_id': 'p2', 'last_point_id': 'p3', 'n_points': 2, 'first_row_id

OK - Test 4 - puente import -> validate -> infer (clusters, selected_fields)


### Test 5 - precondición fatal: trazas no validadas con configuración por defecto

Este test verifica el contrato por defecto de OP-16: si `require_validated_traces=True` y el dataset no viene validado, la inferencia debe abortar.

In [14]:
traces = make_trace_dataset(
    make_trace_points_rich_df(),
    validated=False,
    dataset_id="trace_points_unvalidated_default",
    events=[{"op": "import_traces"}],
)
trip_schema = make_trip_schema_min()

try:
    infer_trips_from_traces(
        traces,
        trip_schema,
        options=make_points_options(),  # require_validated_traces=True por default
    )
    raise AssertionError("Se esperaba InferenceError por trazas no validadas.")
except InferenceError as exc:
    assert exc.code == "INF.PRECONDITION.TRACES_NOT_VALIDATED"

show_ok("Test 5 - precondición fatal de trazas no validadas")

OK - Test 5 - precondición fatal de trazas no validadas


### Test 6 - bypass explícito de validación previa

Aquí se prueba la ruta controlada donde se permite inferir sobre trazas no validadas, pero dejando evidencia explícita del bypass.

In [15]:
traces = make_trace_dataset(
    make_trace_points_rich_df(),
    validated=False,
    dataset_id="trace_points_unvalidated_bypass",
    events=[{"op": "import_traces"}],
)
trip_schema = make_trip_schema_min()

trip_dataset, report = infer_trips_from_traces(
    traces,
    trip_schema,
    options=make_points_options(require_validated_traces=False),
)

assert len(trip_dataset.data) == 7
assert trip_dataset.metadata["is_validated"] is False
assert trip_dataset.metadata["events"][0]["parameters"]["validation_bypass_used"] is True
assert_has_code(report, "INF.PRECONDITION.VALIDATION_BYPASS_USED")

show_ok("Test 6 - bypass explícito de validación previa")

OK - Test 6 - bypass explícito de validación previa


### Test 7 - cierre categórico del output con extensión controlada de dominio

Este caso prueba propagación categórica hacia el `TripSchema` de salida usando un dominio base no vacío pero extendable.  
Se espera que categorías fuera del dominio base extiendan el dominio efectivo en `schema_effective`.

Nota práctica: en la implementación actual conviene pasar mappings vacíos explícitos para los campos categóricos propagados, porque eso fuerza el cierre categórico del output sin alterar valores.

In [16]:
traces = make_trace_dataset(
    make_trace_points_rich_df(),
    validated=True,
    dataset_id="trace_points_domain_extension",
)
trip_schema = make_trip_schema_rich_extendable()

trip_dataset, report = infer_trips_from_traces(
    traces,
    trip_schema,
    options=make_points_options(
        propagate_trace_fields={"poi_cat": "both", "location_ref": "both"}
    ),
    value_correspondence={
        "origin_poi_cat": {},
        "destination_poi_cat": {},
    },
    provenance={"case": "test_7_domain_extension"},
)

assert_has_code(report, "DOM.EXTENSION.APPLIED")
assert "origin_poi_cat" in trip_dataset.data.columns
assert "destination_poi_cat" in trip_dataset.data.columns

origin_domain = trip_dataset.schema_effective.domains_effective["origin_poi_cat"]
dest_domain = trip_dataset.schema_effective.domains_effective["destination_poi_cat"]

assert origin_domain["extendable"] is True
assert dest_domain["extendable"] is True
assert set(origin_domain["values"]).issuperset({"home", "work", "food", "education", "leisure"})
assert set(dest_domain["values"]).issuperset({"home", "work", "food", "education", "leisure"})

show_ok("Test 7 - extensión controlada de dominio en output categórico")

OK - Test 7 - extensión controlada de dominio en output categórico


### Test 8 - `strict_domains=True` con dominio no extendable

Este test cubre la ruta donde el `TripSchema` declara un dominio categórico no extendable y la inferencia encuentra valores fuera de dominio.  
En esa situación, con `strict_domains=True`, se espera `InferenceError`.

In [17]:
traces = make_trace_dataset(
    make_trace_points_rich_df(),
    validated=True,
    dataset_id="trace_points_domain_blocked",
)
trip_schema = make_trip_schema_rich_blocked()

try:
    infer_trips_from_traces(
        traces,
        trip_schema,
        options=make_points_options(
            strict_domains=True,
            propagate_trace_fields={"poi_cat": "both"},
        ),
        value_correspondence={
            "origin_poi_cat": {},
            "destination_poi_cat": {},
        },
        provenance={"case": "test_8_strict_domains"},
    )
    raise AssertionError("Se esperaba InferenceError por dominio no extendable con strict_domains=True.")
except InferenceError as exc:
    assert exc.code == "DOM.STRICT.OUT_OF_DOMAIN_ABORT"

show_ok("Test 8 - strict_domains con dominio bloqueado")

OK - Test 8 - strict_domains con dominio bloqueado


### Test 9 - ruta degradada: thresholds dejan 0 viajes, pero la operación sigue siendo interpretable

Este caso es importante porque verifica la ruta donde no se obtienen viajes materializados, pero igual se construyen dataset vacío, evento y reporte coherentes.

In [18]:
traces = make_trace_dataset(
    make_trace_points_rich_df(),
    validated=True,
    dataset_id="trace_points_zero_output",
)
trip_schema = make_trip_schema_min()

trip_dataset, report = infer_trips_from_traces(
    traces,
    trip_schema,
    options=make_points_options(
        max_time_delta_s=60.0,   # todos los pares reales exceden 60 s
        min_distance_m=None,
    ),
    provenance={"case": "test_9_zero_trips"},
)

assert len(trip_dataset.data) == 0
assert report.summary["n_trips_out"] == 0
assert trip_dataset.metadata["events"][0]["op"] == "infer_trips"
assert trip_dataset.metadata["is_validated"] is False
assert_has_code(report, "INF.CANDIDATES.DROPPED_MAX_TIME_DELTA")
assert_has_code(report, "INF.WARN.ZERO_TRIPS")

show_ok("Test 9 - ruta degradada con 0 viajes")

OK - Test 9 - ruta degradada con 0 viajes


## Resumen de cobertura que te queda con esta batería

Con estos tests de integración estás cubriendo simultáneamente:

- ambos modos de inferencia;
- fixtures ricas en filas y columnas;
- tests puente reales con `import_traces` y `validate_traces`;
- variaciones importantes de `InferTripsOptions`;
- variaciones importantes de `ImportTraceOptions`;
- propagación de campos;
- cierre de H3;
- metadata, evento, provenance y `schema_effective`;
- precondiciones fatales;
- bypass controlado;
- extensión categórica;
- `strict_domains`;
- y la ruta degradada con cero viajes.